# Notebook 02: Thực nghiệm phân cụm khách hàng RFM

Notebook này tiếp tục từ bảng `rfm_table.csv` được tạo ở Notebook 01. Mỗi dòng là một khách hàng; các thuộc tính hành vi được đưa vào phân cụm là `Recency`, `Frequency`, `Monetary`. `CustomerID` chỉ dùng để nhận diện khách hàng và đếm số khách trong cụm, không đưa vào tính khoảng cách.

### Luồng xử lý dữ liệu và thực nghiệm

1. Đọc bảng RFM đã làm sạch; kiểm tra đủ ba biến RFM, bỏ dòng thiếu các biến này và dừng nếu có giá trị âm vì bước log không phù hợp với dữ liệu âm.
2. Lấy tối đa 2.000 khách hàng ngẫu nhiên, không hoàn lại, với `random_state=42`. Cùng một mẫu được dùng cho mọi thuật toán để so sánh công bằng và giới hạn thời gian chạy.
3. Biến đổi `log1p` từng biến RFM nhằm nén độ lệch phải của Frequency/Monetary và giữ được giá trị 0; sau đó `StandardScaler` đưa ba biến về cùng thang đo để biến có đơn vị lớn không lấn át khoảng cách.
4. Huấn luyện baseline K-Means (`k=3`), thử K-Means với `k=2..8`, phân cụm phân cấp Ward với `k=2..8`, và DBSCAN trên các cặp `eps`/`min_samples` đã định trước.
5. Tính Silhouette trên cùng 1.000 khách hàng đầu của mẫu (hoặc toàn bộ nếu mẫu nhỏ hơn). Với DBSCAN, bỏ nhãn nhiễu `-1` chỉ khi tính Silhouette và luôn báo riêng tỷ lệ nhiễu. Chỉ số này không có đáp án đúng/sai vì đây là học không giám sát.
6. Chọn cấu hình có Silhouette cao nhất trong khoảng 2–8 cụm, ưu tiên nghiệm có không quá 20% nhiễu nếu có. Gán nhãn mô hình thắng cho mẫu, tổng hợp trung bình/trung vị RFM, so sánh trung vị cụm với trung vị của mẫu và lưu bảng, biểu đồ, nhật ký.

## Chuẩn bị dữ liệu và chạy các thuật toán

Script đọc bảng RFM, lấy mẫu, biến đổi và chuẩn hóa dữ liệu, chạy các cấu hình phân cụm, chấm điểm rồi lưu bảng/biểu đồ/experiment log. Cell này thực hiện toàn bộ phần tính toán.

In [7]:
# Cell này chuẩn bị đường dẫn, xác nhận đầu vào và chạy toàn bộ thí nghiệm.
from pathlib import Path
import subprocess
import sys
import pandas as pd
from IPython.display import display, HTML

PROJECT_NAME = "Online_Retail_II_GiaiDoan1"
SCRIPT_RELATIVE = Path("src") / "run_clustering_experiments.py"
_search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
_candidates = [item for root in _search_roots for item in (root, root / PROJECT_NAME)]
PROJECT_ROOT = next(
    (item for item in _candidates if (item / SCRIPT_RELATIVE).is_file()), None
)

subprocess.run(
    [sys.executable, str(PROJECT_ROOT / SCRIPT_RELATIVE)],
    cwd=PROJECT_ROOT,
    check=True,
)

# Căn giữa bảng kết quả trong notebook.
display(HTML("""<style>
.dataframe { margin-left: auto !important; margin-right: auto !important; }
.dataframe th, .dataframe td { text-align: center !important; }
</style>"""))


## 1. So sánh cấu hình và thuật toán

Tạo nhãn cụm trên cùng ma trận RFM đã log và chuẩn hóa: một baseline K-Means `k=3`; bảy cấu hình K-Means (`k=2..8`); bảy cấu hình Ward; và 18 cấu hình DBSCAN. Với mỗi cấu hình, script ghi số cụm, kích thước từng cụm, số/tỷ lệ điểm nhiễu và Silhouette trên cùng tập đánh giá. Inertia được ghi cho K-Means nhưng chỉ có ý nghĩa so sánh giữa các giá trị `k` trong cùng thuật toán.

In [8]:
TABLES = PROJECT_ROOT / 'outputs' / 'tables'
comparison = pd.read_csv(TABLES / 'clustering_model_comparison.csv')
display(comparison.sort_values('silhouette_excluding_noise', ascending=False, na_position='last'))


,model,parameters,clusters_excluding_noise,noise_points,noise_percent,silhouette_excluding_noise,inertia,cluster_sizes,evaluated_customers,evaluation_note
1,KMeans_tuned,n_clusters=2; init=k-means++; n_init=20; rando...,2,0,0.00,0.424588,2855.724477,"{""0"": 1162, ""1"": 838}",2000,Silhouette trên điểm không nhiễu; không tính đ...
3,KMeans_tuned,n_clusters=4; init=k-means++; n_init=20; rando...,4,0,0.00,0.366071,1630.053340,"{""0"": 475, ""1"": 410, ""2"": 428, ""3"": 687}",2000,Silhouette trên điểm không nhiễu; không tính đ...
0,KMeans_baseline,n_clusters=3; init=k-means++; n_init=20; rando...,3,0,0.00,0.356581,2087.309634,"{""0"": 828, ""1"": 417, ""2"": 755}",2000,Silhouette trên điểm không nhiễu; không tính đ...
2,KMeans_tuned,n_clusters=3; init=k-means++; n_init=20; rando...,3,0,0.00,0.356581,2087.309634,"{""0"": 828, ""1"": 417, ""2"": 755}",2000,Silhouette trên điểm không nhiễu; không tính đ...
4,KMeans_tuned,n_clusters=5; init=k-means++; n_init=20; rando...,5,0,0.00,0.339272,1350.838038,"{""0"": 376, ""1"": 581, ""2"": 442, ""3"": 449, ""4"": ...",2000,Silhouette trên điểm không nhiễu; không tính đ...
17,DBSCAN,eps=0.25; min_samples=20; metric=euclidean,8,1288,64.40,0.335453,NaN,"{""-1"": 1288, ""0"": 344, ""1"": 49, ""2"": 162, ""3"":...",2000,Silhouette trên điểm không nhiễu; không tính đ...
8,Hierarchical_Ward,n_clusters=2; linkage=ward,2,0,0.00,0.332902,NaN,"{""0"": 1081, ""1"": 919}",2000,Silhouette trên điểm không nhiễu; không tính đ...
5,KMeans_tuned,n_clusters=6; init=k-means++; n_init=20; rando...,6,0,0.00,0.331906,1175.403456,"{""0"": 259, ""1"": 292, ""2"": 425, ""3"": 162, ""4"": ...",2000,Silhouette trên điểm không nhiễu; không tính đ...
6,KMeans_tuned,n_clusters=7; init=k-means++; n_init=20; rando...,7,0,0.00,0.319175,1042.693315,"{""0"": 254, ""1"": 255, ""2"": 299, ""3"": 390, ""4"": ...",2000,Silhouette trên điểm không nhiễu; không tính đ...
23,DBSCAN,eps=0.45; min_samples=20; metric=euclidean,2,194,9.70,0.316413,NaN,"{""-1"": 194, ""0"": 510, ""1"": 1296}",2000,Silhouette trên điểm không nhiễu; không tính đ...


## 2. Mô hình được chọn theo quy tắc đã nêu

Sau khi tạo bảng so sánh, các cấu hình không tạo được ít nhất hai cụm hợp lệ bị loại khỏi bước chọn. Trong các nghiệm có 2–8 cụm, ưu tiên nghiệm có tỷ lệ nhiễu không quá 20%; nếu không có nghiệm nào đạt điều kiện này thì quy tắc nới ngưỡng như mô tả trong bảng. Cuối cùng chọn Silhouette cao nhất (nếu bằng nhau thì ưu tiên ít nhiễu hơn).

In [9]:
selected = pd.read_csv(TABLES / 'selected_clustering_model.csv')
display(selected)


,selected_model,parameters,selection_metric,selection_score,clusters_excluding_noise,noise_percent,sample_size,sampling,preprocessing,selection_rule,caveat
0,KMeans_tuned,n_clusters=2; init=k-means++; n_init=20; rando...,silhouette_excluding_noise,0.424588,2,0.0,2000,Simple random sample without replacement; rand...,"log1p(Recency, Frequency, Monetary), then Stan...",Highest silhouette among 2-8 clusters and <=20...,Exploratory model selection on a fixed evaluat...


## 3. Hồ sơ khách hàng theo cụm: trung bình và trung vị

- Sau khi chọn cấu hình, nhãn cụm được gắn lại vào từng khách hàng trong mẫu. Nhóm theo nhãn cụm rồi tính số khách hàng, trung bình, trung vị của Recency/Frequency/Monetary và tổng Monetary. Các điểm DBSCAN có nhãn `-1` không được tính vào hồ sơ cụm vì chúng không thuộc cụm nào.

- Recency tính bằng ngày kể từ lần mua gần nhất nên thấp hơn là mua gần đây hơn; Frequency là số hóa đơn, Monetary là tổng chi tiêu (GBP), hai biến này cao hơn nghĩa là hoạt động/chi tiêu lớn hơn. Trung bình nhạy với khách hàng chi tiêu rất lớn; trung vị thường đại diện điển hình tốt hơn khi phân phối lệch. `Monetary_sum` mô tả tổng doanh thu của nhóm trong mẫu, không phải giá trị trung bình mỗi khách.

In [10]:
profiles = pd.read_csv(TABLES / 'selected_model_cluster_profiles.csv')
display(profiles)


,cluster,customers,Recency_median,Recency_mean,Frequency_median,Frequency_mean,Monetary_median,Monetary_mean,Monetary_sum
0,0,1162,306.0,306.534423,2.0,2.067986,412.665,616.725872,716635.463
1,1,838,24.0,48.232697,8.0,12.636038,2880.550,5857.184496,4908320.608


## 4. So sánh trung vị cụm với trung vị toàn mẫu

- Từ cùng mẫu 2.000 khách hàng, tính trung vị chung của mỗi biến RFM. Sau đó lấy trung vị của từng cụm so với trung vị chung và gắn nhãn “cao hơn”, “thấp hơn” hoặc “bằng”. Bảng này chuyển các con số ở phần 3 thành mô tả tương đối dễ đọc.

- Cụm 0 có Recency cao hơn nhưng Frequency và Monetary thấp hơn trung vị mẫu, gợi ý nhóm ít hoạt động gần đây và chi tiêu thấp trong mẫu. Cụm 1 có Recency thấp hơn, Frequency và Monetary cao hơn, gợi ý nhóm mua gần đây và chi tiêu nhiều hơn. Đây chỉ là mô tả của mẫu, không phải nhãn khách hàng chắc chắn hay quan hệ nhân quả.

In [11]:
interpretation = pd.read_csv(TABLES / 'cluster_interpretation.csv')
display(interpretation)

print((TABLES / 'cluster_interpretation.md').read_text(encoding='utf-8'))


,cluster,customers,Recency_median,Recency_mean,Frequency_median,Frequency_mean,Monetary_median,Monetary_mean,Monetary_sum,Recency_vs_sample_median,Frequency_vs_sample_median,Monetary_vs_sample_median
0,0,1162,306.0,306.534423,2.0,2.067986,412.665,616.725872,716635.463,cao hơn trung vị mẫu,thấp hơn trung vị mẫu,thấp hơn trung vị mẫu
1,1,838,24.0,48.232697,8.0,12.636038,2880.550,5857.184496,4908320.608,thấp hơn trung vị mẫu,cao hơn trung vị mẫu,cao hơn trung vị mẫu


# Diễn giải cụm từ mẫu RFM

Mô hình được chọn theo silhouette: **KMeans_tuned** (n_clusters=2; init=k-means++; n_init=20; random_state=42).
Silhouette trên tập đánh giá cố định: **0.4246**; 2 cụm; noise DBSCAN 0.00%.
Trung vị toàn mẫu: Recency=92.00 ngày, Frequency=3.00 hóa đơn, Monetary=915.03 GBP.

Các mô tả dưới đây so sánh trung vị của từng cụm với trung vị mẫu chung; đây là diễn giải mô tả, không phải nhãn chân lý hay bằng chứng nhân quả.

- **Cụm 0** (1,162 khách hàng trong mẫu): Recency 306.00 ngày (cao hơn trung vị mẫu), Frequency 2.00 hóa đơn (thấp hơn trung vị mẫu), Monetary 412.66 GBP (thấp hơn trung vị mẫu); doanh thu gộp trong cụm 716635.46 GBP.
- **Cụm 1** (838 khách hàng trong mẫu): Recency 24.00 ngày (thấp hơn trung vị mẫu), Frequency 8.00 hóa đơn (cao hơn trung vị mẫu), Monetary 2880.55 GBP (cao hơn trung vị mẫu); doanh thu gộp trong cụm 4908320.61 GBP.

## Giới hạn

Kết quả và hồ sơ chỉ đại diện cho mẫu ngẫu nhiên cố định tối đa 2.000 khách hàng. Cần kiểm tra độ ổn đị